# Semantra Backend Adapter — Example

This notebook shows how to use the `semantra-backend-adapter` package to
expose the real Semantra FastAPI backend as concrete implementations of the
`semantra-core` protocols.

If the backend is not importable in the current environment, each adapter
falls back to the in-memory reference implementation from `semantra-core`.

In [ ]:
# Install both packages in editable mode (run once)
# !pip install -e ../semantra-core
# !pip install -e ../semantra-backend-adapter

In [ ]:
from semantra_core.models.schema import (
    DatasetHandle,
    SchemaProfile,
    ColumnProfile,
)
from semantra_backend_adapter import (
    BackendContext,
    create_backend_adapters,
    BackendMappingEngine,
    BackendKnowledgeBase,
    BackendLLMService,
    BackendConnector,
)

## 1. Build a default context

`create_default_context()` will try to import the real backend and populate
the context with its settings and database session factory. If the backend
is not importable, it raises a `RuntimeError` — in that case, we construct a
blank context for demonstration.

In [ ]:
try:
    ctx = create_default_context()
    print("Loaded real backend context.")
except RuntimeError as e:
    print(f"Backend not importable: {e}")
    print("Falling back to an empty context (adapters will use stubs).")
    ctx = BackendContext()

## 2. Build a source and target

In [ ]:
source = DatasetHandle(
    dataset_id="src1",
    dataset_name="customer_src",
    schema_profile=SchemaProfile(
        dataset_id="src1",
        dataset_name="customer_src",
        row_count=50,
        columns=[
            ColumnProfile(
                name="cust_id",
                normalized_name="cust_id",
                dtype="str",
                null_ratio=0.0,
                unique_ratio=1.0,
                non_null_count=50,
            )
        ],
    ),
)
target = SchemaProfile(
    dataset_id="tgt1",
    dataset_name="customer_tgt",
    row_count=0,
    columns=[
        ColumnProfile(
            name="customer_key",
            normalized_name="customer_key",
            dtype="str",
            null_ratio=0.0,
            unique_ratio=0.0,
            non_null_count=0,
        )
    ],
)
print("Source and target ready.")

## 3. Use the factory to build all adapters at once

In [ ]:
adapters = create_backend_adapters(context=ctx, dataset_id="src1")
print("Available adapters:", list(adapters.keys()))

## 4. Call the mapping engine through the adapter

In [ ]:
candidates = adapters["engine"].map_source_to_target(source, target)
print(f"Got {len(candidates)} candidates from the backend mapping engine.")

## 5. Call the LLM service through the adapter

In [ ]:
result = adapters["llm"].validate_mapping(
    source_field="cust_id",
    candidate_targets=["customer_key", "id"],
    context={"description": "primary key"},
)
print("LLM validation result:", result)

## 6. Use the connector (if a dataset was provided)

In [ ]:
if "connector" in adapters:
    schema = adapters["connector"].fetch_schema()
    print("Fetched schema from connector:", schema.dataset_name)
else:
    print("No connector was created (no dataset_id provided).")